# Домашнее задание 3 — Агент-аналитик системных промптов (MCP + Skills)

### Задания

1. **Файловые инструменты агента** — реализовать инструменты для сохранения/чтения результатов анализа
2. **Пайплайн: агенты с tool loop** — реализовать агента и оркестратор пайплайна

### Критерии оценки (10 баллов)

| # | Критерий | Балл |
|---|----------|------|
| 1 | Реализованы все 5 файловых инструментов (`save_analysis`, `list_analyses`, `read_analysis`, `save_report`, `read_report`) + tool schemas + `LOCAL_TOOLS` | 1 |
| 2 | Написаны все 4 skill-файла (`skills.md`, `prompt_analysis.md`, `analysis_report.md`, `prompt_generator.md`) с двухуровневой структурой (Level 1 — метаданные в `skills.md`, Level 2 — полная процедура) | 1 |
| 3 | `prompt_analysis.md` содержит ≥5 критериев оценки промпта (Role/Persona, Few-shot, Chain-of-thought, Output format, Guardrails и т.д.) | 1 |
| 4 | Реализован `run_agent` с корректным tool loop: маршрутизация вызовов (MCP / `read_skill` / локальные), трассировка через `AgentTrace`, чистый контекст на каждый вызов | 1 |
| 5 | Сценарий `prompt_analysis` работает: агент загружает skill → читает файл из GitHub → анализирует → сохраняет через `save_analysis` | 1 |
| 6 | Сценарий `analysis_report` работает: агент читает все анализы → формирует сводный отчёт → сохраняет через `save_report` | 1 |
| 7 | Сценарий `prompt_generator` работает: агент читает отчёт → генерирует новый системный промпт → сохраняет результат | 1 |
| 8 | Progressive disclosure реализован как для skills, так и для MCP-инструментов | 1 |
| 9–10 | Сценарий `prompt_analysis` реализован через субагента (отдельный агентный вызов для каждого файла, а не один монолитный контекст) | 2 |

---
## 0. Подготовка

In [17]:
import warnings

warnings.filterwarnings("ignore")

import sys
import json
import asyncio
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import nest_asyncio

nest_asyncio.apply()

from src.config import settings
from openai import OpenAI

In [18]:
# ============================================================
# Настройка LLM-клиентов
# ============================================================

client = OpenAI(
    base_url="https://api.polza.ai/api/v1",
    api_key=settings.polza_ai_api_key,
)

MODEL = "openai/gpt-4o"

---
## 0.1 Утилиты для трассировки агента

Код из практики — необходим для выполнения заданий.

In [19]:
# ============================================================
# Утилиты для трассировки агента (из практики 3)
# ============================================================
from dataclasses import dataclass, field


def _extract_usage(response) -> dict:
    """Извлечь токены из response.usage."""
    usage = getattr(response, "usage", None)
    if not usage:
        return {"prompt_tokens": 0, "cached_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "context_size": 0}

    prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
    completion_tokens = getattr(usage, "completion_tokens", 0) or 0
    total_tokens = getattr(usage, "total_tokens", 0) or 0

    cached_tokens = getattr(usage, "prompt_cache_hit_tokens", 0) or 0
    if not cached_tokens:
        details = getattr(usage, "prompt_tokens_details", None)
        if details:
            cached_tokens = getattr(details, "cached_tokens", 0) or 0

    context_size = prompt_tokens + cached_tokens
    return {
        "prompt_tokens": prompt_tokens,
        "cached_tokens": cached_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "context_size": context_size,
    }


@dataclass
class AgentStep:
    """Один шаг агента — для трассировки."""

    step_number: int
    tool_calls: list[dict] = field(default_factory=list)
    observations: list[dict] = field(default_factory=list)
    prompt_tokens: int = 0
    cached_tokens: int = 0
    context_size: int = 0
    context_size_delta: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    final_answer: str | None = None


@dataclass
class AgentTrace:
    """Полный лог работы агента."""

    question: str
    steps: list[AgentStep] = field(default_factory=list)
    total_steps: int = 0
    prompt_tokens_total: int = 0
    completion_tokens_total: int = 0
    total_tokens_total: int = 0
    latest_context_size: int = 0
    max_context_size: int = 0
    final_answer: str = ""

    def summary(self) -> str:
        lines = [
            f"Вопрос: {self.question}",
            f"Шагов: {self.total_steps}",
            f"Prompt tokens (биллинг): {self.prompt_tokens_total}",
            f"Completion tokens: {self.completion_tokens_total}",
            f"Всего tokens: {self.total_tokens_total}",
            "",
        ]
        for s in self.steps:
            lines.append(f"--- Шаг {s.step_number} ---")
            if s.total_tokens:
                lines.append(
                    f"  \U0001f522 context={s.context_size} (\u0394 {s.context_size_delta:+}), "
                    f"completion={s.completion_tokens}"
                )
            for tc in s.tool_calls:
                lines.append(f"  \U0001f527 {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)[:80]})")
            for obs in s.observations:
                text = str(obs)
                lines.append(f"  \U0001f4e1 {text[:120]}{'...' if len(text) > 120 else ''}")
            if s.final_answer:
                lines.append(f"  \U0001f4ac {s.final_answer[:300]}")
        lines.append(f"\n{'=' * 50}\n\U0001f3af Финальный ответ:\n{self.final_answer[:800]}")
        return "\n".join(lines)

---
## 0.2 Источник данных — GitHub MCP Server

Подключаемся к официальному GitHub MCP-серверу для доступа к репозиторию с системными промптами.

Репозиторий [`asgeirtj/system_prompts_leaks`](https://github.com/asgeirtj/system_prompts_leaks) — коллекция реальных системных промптов (ChatGPT, Claude, Gemini, Codex и др.)

### Предварительная настройка

#### 1. Node.js и npx

**npx** — утилита из экосистемы Node.js для запуска npm-пакетов без установки.

Проверить установку:
```bash
node --version   # должно быть v18+ 
npx --version    # должно быть 9+
```

#### 2. GitHub Personal Access Token

Создаётся в [Settings → Developer settings → Personal access tokens → Tokens (classic)](https://github.com/settings/tokens).

Минимальные права: `public_repo`.

Добавьте токен в файл `.env` в корне проекта:
```
GITHUB_TOKEN=ghp_ваш_токен_здесь
```

In [20]:
# ============================================================
# Подключение к GitHub MCP Server через stdio
# ============================================================
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Параметры подключения — запускаем официальный GitHub MCP-сервер
github_server_params = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-github"],
    env={
        **os.environ,
        "GITHUB_PERSONAL_ACCESS_TOKEN": settings.github_token,
    },
)

In [21]:
# ============================================================
# Discovery: подключаемся к GitHub MCP и узнаём его возможности
# ============================================================


async def discover_github_server():
    """Подключиться к GitHub MCP-серверу и получить список инструментов."""
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("\u2705 Подключение к GitHub MCP установлено!\n")

                # Discovery: Tools
                tools_response = await session.list_tools()
                print(f"\U0001f527 Инструменты ({len(tools_response.tools)}):")
                print("-" * 55)
                for tool in tools_response.tools:
                    desc = (tool.description or "")[:80]
                    print(f"  \u2022 {tool.name}")
                    print(f"    {desc}...")
                    print()

                return tools_response.tools


github_tools = asyncio.get_event_loop().run_until_complete(discover_github_server())

✅ Подключение к GitHub MCP установлено!

🔧 Инструменты (26):
-------------------------------------------------------
  • create_or_update_file
    Create or update a single file in a GitHub repository...

  • search_repositories
    Search for GitHub repositories...

  • create_repository
    Create a new GitHub repository in your account...

  • get_file_contents
    Get the contents of a file or directory from a GitHub repository...

  • push_files
    Push multiple files to a GitHub repository in a single commit...

  • create_issue
    Create a new issue in a GitHub repository...

  • create_pull_request
    Create a new pull request in a GitHub repository...

  • fork_repository
    Fork a GitHub repository to your account or specified organization...

  • create_branch
    Create a new branch in a GitHub repository...

  • list_commits
    Get list of commits of a branch in a GitHub repository...

  • list_issues
    List issues in a GitHub repository with filtering options...

 

---
## 0.3 Skills — процедурные навыки агента

В практике мы создали skills для космического ассистента.  
Здесь используем **три навыка для анализа промптов**:

| Навык | Файл | Назначение |
|-------|------|------------|
| `prompt_analysis` | `hw_3_skills/prompt_analysis.md` | Пошаговый разбор системного промпта: секции, persona, приёмы, оценки |
| `analysis_report` | `hw_3_skills/analysis_report.md` | Формирование сводного отчёта с таблицами, цитатами и рекомендациями |
| `prompt_generator` | `hw_3_skills/prompt_generator.md` | Генерация нового промпта на основе выявленных паттернов |

### Skill Chaining

В практике skills были **независимые**. Здесь они образуют **цепочку** (chain):

```
prompt_analysis ──(структурированный разбор)──→ analysis_report
                                                       │
                                              (отчёт + рекомендации)
                                                       │
                                                       ▼
                                               prompt_generator
```

**Выход одного skill = вход следующего.** Агент сам решает, когда переключиться.

In [22]:
# ============================================================
# Реестр Skills — загружаем все skill-файлы из hw_3_skills/
# ============================================================

from pydantic import BaseModel, Field


class SkillInfo(BaseModel):
    """Метаданные одного skill-файла."""

    name: str = Field(description="имя файла (без .md)")
    title: str = Field(description="Заголовок skill")
    trigger_description: str = Field(description="Когда активировать (из секции 'Когда активировать')")
    content: str = Field(description="Полный текст skill")


def load_skills(skills_directory: Path) -> dict[str, SkillInfo]:
    """Загрузить все skill-файлы из директории (кроме skills.md)."""
    skills = {}

    for skill_file in sorted(skills_directory.glob("*.md")):
        if skill_file.name == "skills.md":
            continue

        with open(skill_file, "r", encoding="utf-8") as f:
            content = f.read()

        lines = content.split("\n")
        title = lines[0].replace("# Skill: ", "").strip() if lines else skill_file.stem

        trigger = ""
        in_trigger = False
        for line in lines:
            if "Когда активировать" in line:
                in_trigger = True
                continue
            if in_trigger:
                if line.startswith("##") and "Когда" not in line:
                    break
                trigger += line + "\n"

        name = skill_file.stem
        skills[name] = SkillInfo(
            name=name,
            title=title,
            trigger_description=trigger.strip(),
            content=content,
        )

    return skills


def load_skills_metadata(skills_directory: Path) -> str:
    """Загрузить skills.md — Level 1 метаданные для system prompt."""
    skills_md = skills_directory / "skills.md"
    if skills_md.exists():
        with open(skills_md, "r", encoding="utf-8") as f:
            return f.read()
    return "\u26a0\ufe0f Файл skills.md не найден в директории skills."


# --- Загрузка ---
skills_dir = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_skills"
SKILLS = load_skills(skills_dir)

print(f"\U0001f4c2 Директория skills: {skills_dir}")
print(f"\U0001f4cb Загружено skills: {len(SKILLS)}\n")

for name, info in SKILLS.items():
    print(f"  \u2022 {name}: {info.title}")
    print(f"    Триггер: {info.trigger_description[:80]}...")
    print()

📂 Директория skills: /Users/bogdanmaslennikov/Desktop/Новая папка/HSE-Agent-Systems_2026/lectures/lecture_3/hw_3_skills
📋 Загружено skills: 3

  • analysis_report: Формирование аналитического отчёта
    Триггер: Этот skill активируется, когда:
- Анализ промптов (навык `prompt_analysis`) заве...

  • prompt_analysis: Анализ системного промпта
    Триггер: Этот skill активируется, когда нужно:
- Разобрать системный промпт на составные ...

  • prompt_generator: Генерация системного промпта
    Триггер: Этот skill активируется, когда:
- Аналитический отчёт (навык `analysis_report`) ...



In [23]:
# ============================================================
# Level 1: загружаем skills.md — краткий обзор навыков для system prompt
# ============================================================

skills_metadata = load_skills_metadata(skills_dir)
print("Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):\n")
print(skills_metadata)

Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):

# Skills — Навыки агента-аналитика системных промптов

Набор skills, определяющих **как** агент должен действовать при анализе системных промптов, формировании отчётов и генерации новых промптов.

## Список навыков

| Навык | Файл | Краткое описание |
|-------|------|------------------|
| Анализ промпта | [prompt_analysis.md](prompt_analysis.md) | Разбор системного промпта на составные части: секции, persona, приёмы prompt engineering, оценка качества по 5+ критериям. Результат сохраняется через `save_analysis()` |
| Аналитический отчёт | [analysis_report.md](analysis_report.md) | Агрегация результатов анализа нескольких промптов: матрица критериев, общие паттерны, лучшие практики, сравнительные оценки. Результат сохраняется через `save_report()` |
| Генерация промпта | [prompt_generator.md](prompt_generator.md) | Создание нового системного промпта на основе выявленных лучших практик из аналитического отчёта. Рез

In [24]:
# ============================================================
# Инструмент read_skill — загрузка процедуры по требованию
# ============================================================

READ_SKILL_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "read_skill",
        "description": (
            "Загрузить детальную процедуру (skill) для решения задачи определённого типа. "
            "Вызывай этот инструмент, когда задача пользователя соответствует одному из "
            "доступных skills. После загрузки — строго следуй процедуре шаг за шагом."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": ("Имя skill для загрузки: " + ", ".join(SKILLS.keys())),
                }
            },
            "required": ["name"],
        },
    },
}


def read_skill(name: str) -> str:
    """Загрузить полный контент skill по имени."""
    if name in SKILLS:
        skill = SKILLS[name]
        return (
            f"\u2550\u2550\u2550 SKILL ЗАГРУЖЕН: {skill.title} \u2550\u2550\u2550\n\n"
            f"СТРОГО следуй этой процедуре шаг за шагом.\n\n"
            f"{skill.content}"
        )
    available = ", ".join(SKILLS.keys())
    return f"Skill '{name}' не найден. Доступные skills: {available}"


# --- Тест ---
print("\U0001f527 READ_SKILL_TOOL_SCHEMA:")
print(json.dumps(READ_SKILL_TOOL_SCHEMA, ensure_ascii=False, indent=2)[:500])
print("\n--- Тест: read_skill('prompt_analysis') ---")
print(read_skill("prompt_analysis")[:400] + "...")

🔧 READ_SKILL_TOOL_SCHEMA:
{
  "type": "function",
  "function": {
    "name": "read_skill",
    "description": "Загрузить детальную процедуру (skill) для решения задачи определённого типа. Вызывай этот инструмент, когда задача пользователя соответствует одному из доступных skills. После загрузки — строго следуй процедуре шаг за шагом.",
    "parameters": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string",
          "description": "Имя skill для загрузки: analysis_report, prompt_a

--- Тест: read_skill('prompt_analysis') ---
═══ SKILL ЗАГРУЖЕН: Анализ системного промпта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Анализ системного промпта

## Требуемые инструменты
- get_file_contents (GitHub MCP)
- save_analysis (локальный)

## Когда активировать
Этот skill активируется, когда нужно:
- Разобрать системный промпт на составные части
- Оценить качество и полноту промпта
- Выявить приёмы prompt engineering в...


---
## 0.4 Вспомогательные функции пайплайна

Код из практики — утилиты для обхода репозитория и конвертации MCP-инструментов.

In [25]:

def format_mcp_tools(mcp_tools) -> list[dict]:
    """Конвертировать MCP tools в OpenAI function-calling формат."""
    formatted = []
    for tool in mcp_tools:
        schema = tool.inputSchema if tool.inputSchema else {"type": "object", "properties": {}}
        formatted.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameters": schema,
                },
            }
        )
    return formatted


---
## Задание 1: Файловые инструменты агента

Агент работает в несколько этапов, и между этапами ему нужно **сохранять и читать результаты** через файловую систему.

### Что нужно реализовать

| Инструмент | Назначение | Когда используется |
|------------|-----------|--------------------|
| `save_analysis(name, content)` | Сохранить анализ одного промпта в файл | После анализа каждого промпта |
| `list_analyses()` | Получить список всех сохранённых анализов | Перед формированием отчёта |
| `read_analysis(name)` | Прочитать конкретный анализ | При формировании отчёта |
| `save_report(name, content)` | Сохранить сводный отчёт / промпт | После формирования отчёта / промпта |
| `read_report(name)` | Прочитать сохранённый отчёт | Перед генерацией промпта |

### Файловая структура

```
hw_3_output/
  analyses/       ← save_analysis / read_analysis / list_analyses
    OpenAI_Codex.md
    Anthropic_Claude.md
    ...
  reports/         ← save_report / read_report
    analysis_report.md
    generated_prompt.md
```

### Подсказки
- Не забудьте добавить tool schemas в формате OpenAI function-calling
- Соберите все локальные инструменты в маппинг `LOCAL_TOOLS`

In [26]:
# ============================================================
# Инструменты агента
# ============================================================

# --- Директории для результатов ---
ANALYSES_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "analyses"
REPORTS_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "reports"

ANALYSES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Директория анализов: {ANALYSES_DIR}")
print(f"📂 Директория отчётов:  {REPORTS_DIR}")


def save_analysis(name: str, content: str) -> str:
    """Сохранить сводку по анализу одного промпта."""
    path = ANALYSES_DIR / f"{name}.md"
    path.write_text(content, encoding="utf-8")
    return f"✅ Анализ сохранён: {path.name} ({len(content)} символов)"


def list_analyses() -> str:
    """Получить список всех сохранённых анализов."""
    files = sorted(ANALYSES_DIR.glob("*.md"))
    if not files:
        return "❌ Нет сохранённых анализов. Сначала запусти сценарий prompt_analysis."
    lines = [f"📋 Сохранённые анализы ({len(files)}):"]
    for f in files:
        size = f.stat().st_size
        lines.append(f"  • {f.stem} ({size} байт)")
    return "\n".join(lines)


def read_analysis(name: str) -> str:
    """Прочитать сохранённый анализ промпта по имени."""
    path = ANALYSES_DIR / f"{name}.md"
    if path.exists():
        content = path.read_text(encoding="utf-8")
        return f"═══ АНАЛИЗ: {name} ═══\n\n{content}"
    files = sorted(ANALYSES_DIR.glob("*.md"))
    available = ", ".join(f.stem for f in files) if files else "нет"
    return f"❌ Анализ '{name}' не найден. Доступные: {available}"


def save_report(name: str, content: str) -> str:
    """Сохранить отчёт или сгенерированный промпт."""
    path = REPORTS_DIR / f"{name}.md"
    path.write_text(content, encoding="utf-8")
    return f"✅ Отчёт сохранён: {path.name} ({len(content)} символов)"


def read_report(name: str) -> str:
    """Прочитать сохранённый отчёт."""
    path = REPORTS_DIR / f"{name}.md"
    if path.exists():
        return path.read_text(encoding="utf-8")
    files = sorted(REPORTS_DIR.glob("*.md"))
    available = ", ".join(f.stem for f in files) if files else "нет"
    return f"❌ Отчёт '{name}' не найден. Доступные: {available}"


async def list_repo_files(session) -> list[dict]:
    """
    Обойти репозиторий через MCP и собрать список промпт-файлов.
    Возвращает [{"path": "OpenAI/Codex.md", "name": "Codex.md", "folder": "OpenAI", "size": ...}]
    """
    REPO = {"owner": "asgeirtj", "repo": "system_prompts_leaks"}
    root_result = await session.call_tool("get_file_contents", {**REPO, "path": ""})
    root_data = json.loads(root_result.content[0].text)
    files = []
    for item in root_data:
        if item.get("type") != "dir":
            continue
        folder = item["name"]
        try:
            dir_result = await session.call_tool("get_file_contents", {**REPO, "path": folder})
            dir_data = json.loads(dir_result.content[0].text)
            for f in dir_data:
                if f.get("type") == "file" and f["name"].endswith(".md"):
                    files.append({"path": f["path"], "name": f["name"], "folder": folder, "size": f.get("size", 0)})
        except Exception:
            continue
    return files


📂 Директория анализов: /Users/bogdanmaslennikov/Desktop/Новая папка/HSE-Agent-Systems_2026/lectures/lecture_3/hw_3_output/analyses
📂 Директория отчётов:  /Users/bogdanmaslennikov/Desktop/Новая папка/HSE-Agent-Systems_2026/lectures/lecture_3/hw_3_output/reports


In [27]:
# ============================================================
# Tool schemas в формате OpenAI function-calling
# ============================================================

FILE_TOOL_SCHEMAS = [
    {"type": "function", "function": {"name": "save_analysis", "description": "Сохранить анализ одного системного промпта. Вызывай ПОСЛЕ завершения анализа — не возвращай анализ как текст.", "parameters": {"type": "object", "properties": {"name": {"type": "string", "description": "Имя файла без расширения"}, "content": {"type": "string", "description": "Полный текст анализа в Markdown"}}, "required": ["name", "content"]}}},
    {"type": "function", "function": {"name": "list_analyses", "description": "Получить список всех сохранённых анализов. Вызывай перед формированием отчёта.", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "read_analysis", "description": "Прочитать сохранённый анализ по имени.", "parameters": {"type": "object", "properties": {"name": {"type": "string", "description": "Имя анализа без расширения"}}, "required": ["name"]}}},
    {"type": "function", "function": {"name": "save_report", "description": "Сохранить сводный отчёт или сгенерированный промпт. Вызывай ПОСЛЕ завершения — не возвращай как текст.", "parameters": {"type": "object", "properties": {"name": {"type": "string", "description": "Имя файла без расширения"}, "content": {"type": "string", "description": "Полный текст в Markdown"}}, "required": ["name", "content"]}}},
    {"type": "function", "function": {"name": "read_report", "description": "Прочитать сохранённый отчёт по имени.", "parameters": {"type": "object", "properties": {"name": {"type": "string", "description": "Имя отчёта без расширения"}}, "required": ["name"]}}},
]

LOCAL_TOOLS = {
    "read_skill": read_skill,
    "save_analysis": save_analysis,
    "list_analyses": list_analyses,
    "read_analysis": read_analysis,
    "save_report": save_report,
    "read_report": read_report,
}

print(f"🔧 Локальные инструменты агента ({len(LOCAL_TOOLS)}):")
for name in LOCAL_TOOLS:
    print(f"  • {name}")


🔧 Локальные инструменты агента (6):
  • read_skill
  • save_analysis
  • list_analyses
  • read_analysis
  • save_report
  • read_report


---
## Пайплайн — универсальный агент с tool loop

Реализуйте **одного универсального агента** `run_agent()`, который выполняет разные задачи в зависимости от запроса (`Начни анализ`, `сформируй сводку`, `сгенерируй промпт для такой то задачи`). 
### Подсказки

- За основу tool loop возьмите `run_reactive_agent` из практики 2 (или аналог из практики 3)
- Используйте `AgentTrace` для отслеживания шагов и токенов

In [28]:
# ============================================================
# System prompt агента
# ============================================================

def build_system_prompt(skills_metadata: str, mcp_tools_preview: str) -> str:
    return (
        "Ты агент-аналитик системных промптов. Анализируй промпты из репозитория, "
        "формируй отчёты и генерируй новые промпты на основе лучших практик.\n\n"
        "## Правила работы\n"
        "- Перед выполнением задачи СНАЧАЛА загрузи нужный skill через `read_skill`\n"
        "- НИКОГДА не возвращай анализ как финальный текст — ВСЕГДА сначала вызови `save_analysis`\n"
        "- НИКОГДА не возвращай отчёт как финальный текст — ВСЕГДА сначала вызови `save_report`\n"
        "- Финальный ответ = короткое подтверждение (1-2 предложения), не само содержимое\n\n"
        "## Доступные навыки\n\n"
        + skills_metadata
        + "\n\n## MCP-инструменты (GitHub)\n\n"
        + mcp_tools_preview
        + "\n\nФайловые инструменты (`save_analysis`, `list_analyses`, `read_analysis`, `save_report`, `read_report`) доступны всегда."
    )


def validate_tool_call(func_name: str, args: dict) -> str | None:
    if func_name == "save_analysis":
        if not args.get("name"): return "\u274c save_analysis: параметр 'name' пустой"
        if not args.get("content"): return "\u274c save_analysis: параметр 'content' пустой"
    if func_name == "save_report":
        if not args.get("name"): return "\u274c save_report: параметр 'name' пустой"
        if not args.get("content"): return "\u274c save_report: параметр 'content' пустой"
    if func_name == "read_skill":
        name = args.get("name", "")
        if name not in SKILLS:
            return f"\u274c skill '{name}' не найден. Доступные: {', '.join(SKILLS.keys())}"
    return None


async def run_agent(
    question: str,
    session,
    all_mcp_schemas: dict[str, dict],
    required_tool: str | None = None,
    max_steps: int = 15,
    verbose: bool = True,
) -> AgentTrace:
    trace = AgentTrace(question=question)
    mcp_tool_names = set(all_mcp_schemas.keys())
    called_tools: set[str] = set()

    active_tools = [READ_SKILL_TOOL_SCHEMA] + FILE_TOOL_SCHEMAS
    active_tool_names = {"read_skill"} | {t["function"]["name"] for t in FILE_TOOL_SCHEMAS}
    mcp_preview = "\n".join(f"  \u2022 {n}" for n in mcp_tool_names)

    messages = [
        {"role": "system", "content": build_system_prompt(skills_metadata, mcp_preview)},
        {"role": "user", "content": question},
    ]
    prev_context_size = 0

    for step_num in range(1, max_steps + 1):
        current_step = AgentStep(step_number=step_num)
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=active_tools, tool_choice="auto", temperature=0,
        )
        u = _extract_usage(response)
        current_step.prompt_tokens = u["prompt_tokens"]
        current_step.cached_tokens = u["cached_tokens"]
        current_step.completion_tokens = u["completion_tokens"]
        current_step.total_tokens = u["total_tokens"]
        current_step.context_size = u["context_size"]
        current_step.context_size_delta = current_step.context_size - prev_context_size
        prev_context_size = current_step.context_size
        trace.prompt_tokens_total += current_step.prompt_tokens
        trace.completion_tokens_total += current_step.completion_tokens
        trace.total_tokens_total += current_step.total_tokens
        trace.latest_context_size = current_step.context_size
        trace.max_context_size = max(trace.max_context_size, current_step.context_size)

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            if required_tool and required_tool not in called_tools:
                reminder = (
                    f"\u26a0\ufe0f СТОП. Инструмент `{required_tool}` не был вызван. "
                    f"Ты ОБЯЗАН вызвать `{required_tool}` с результатом своей работы прямо сейчас. "
                    f"Не возвращай результат как текст."
                )
                messages.append({"role": "user", "content": reminder})
                current_step.observations.append(f"[VALIDATOR] {required_tool} not called")
                trace.steps.append(current_step)
                if verbose:
                    print(f"  \u26a0\ufe0f [{step_num}] Валидатор: {required_tool} не вызван")
                continue
            current_step.final_answer = msg.content
            trace.final_answer = msg.content or ""
            trace.steps.append(current_step)
            trace.total_steps = step_num
            if verbose:
                print(f"\n{'=' * 50}")
                print(f"\u2705 Агент завершил работу за {step_num} шагов")
                print(f"\U0001f4ac {trace.final_answer[:300]}")
            return trace

        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments) if isinstance(tool_call.function.arguments, str) else tool_call.function.arguments
            called_tools.add(func_name)
            current_step.tool_calls.append({"name": func_name, "args": args})
            if verbose:
                print(f"  \U0001f527 [{step_num}] {func_name}({json.dumps(args, ensure_ascii=False)[:100]})")

            err = validate_tool_call(func_name, args)
            if err:
                current_step.observations.append(err)
                messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": err})
                continue

            if func_name == "read_skill":
                skill_name = args.get("name", "")
                result_text = read_skill(skill_name)
                if skill_name in SKILLS:
                    newly = [t for t in mcp_tool_names if t not in active_tool_names]
                    for t in newly:
                        active_tools.append(all_mcp_schemas[t])
                        active_tool_names.add(t)
                    if newly and verbose:
                        print(f"  \U0001f513 Разблокированы MCP: {newly}")
            elif func_name in LOCAL_TOOLS:
                try:
                    result_text = LOCAL_TOOLS[func_name]() if func_name == "list_analyses" else LOCAL_TOOLS[func_name](**args)
                except Exception as e:
                    result_text = f"\u274c Ошибка {func_name}: {e}"
            elif func_name in mcp_tool_names:
                try:
                    result = await session.call_tool(func_name, args)
                    result_text = result.content[0].text if result.content else "{}"
                except Exception as e:
                    result_text = json.dumps({"error": str(e)}, ensure_ascii=False)
            else:
                result_text = json.dumps({"error": f"Неизвестный инструмент: {func_name}"}, ensure_ascii=False)

            current_step.observations.append(str(result_text)[:200])
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(result_text)})

        trace.steps.append(current_step)

    trace.final_answer = "\u274c Превышен лимит шагов агента"
    trace.total_steps = max_steps
    return trace


In [29]:
MAX_FILES_TO_ANALYZE = 3

async def run_prompt_analysis_pipeline():
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                tools_response = await session.list_tools()
                all_mcp_schemas = {t["function"]["name"]: t for t in format_mcp_tools(tools_response.tools)}
                print("📂 Получаем список файлов из репозитория...")
                files = await list_repo_files(session)
                files_to_analyze = files[:MAX_FILES_TO_ANALYZE]
                print(f"📋 Найдено {len(files)} файлов, анализируем {len(files_to_analyze)}")
                traces = []
                for i, file_info in enumerate(files_to_analyze, 1):
                    print(f"\n{'─' * 50}")
                    print(f"[{i}/{len(files_to_analyze)}] Анализирую: {file_info['path']}")
                    question = (
                        f"Проанализируй системный промпт из файла '{file_info['path']}' "
                        f"в репозитории asgeirtj/system_prompts_leaks. "
                        f"Сохрани анализ под именем '{file_info['name'].replace('.md', '')}'"
                    )
                    trace = await run_agent(
                        question=question, session=session, all_mcp_schemas=all_mcp_schemas,
                        required_tool="save_analysis", max_steps=12, verbose=True,
                    )
                    traces.append(trace)
                print(f"\n{'=' * 50}")
                print(f"✅ Анализ завершён. Обработано: {len(traces)}")
                print(list_analyses())
                return traces

traces_analysis = asyncio.get_event_loop().run_until_complete(run_prompt_analysis_pipeline())


📂 Получаем список файлов из репозитория...
📋 Найдено 105 файлов, анализируем 3

──────────────────────────────────────────────────
[1/3] Анализирую: Anthropic/claude-code.md
  🔧 [1] read_skill({"name": "prompt_analysis"})
  🔓 Разблокированы MCP: ['get_issue', 'search_users', 'add_issue_comment', 'create_pull_request_review', 'list_commits', 'update_pull_request_branch', 'list_pull_requests', 'search_code', 'create_branch', 'get_pull_request_status', 'get_pull_request_files', 'create_pull_request', 'create_or_update_file', 'push_files', 'create_issue', 'get_pull_request', 'merge_pull_request', 'create_repository', 'update_issue', 'search_issues', 'list_issues', 'fork_repository', 'get_pull_request_reviews', 'get_pull_request_comments', 'get_file_contents', 'search_repositories']
  🔧 [2] get_file_contents({"owner": "asgeirtj", "repo": "system_prompts_leaks", "path": "Anthropic/claude-code.md"})
  🔧 [3] save_analysis({"name": "claude-code", "content": "# Анализ промпта: claude-code\n\n## 

In [30]:
async def run_analysis_report_pipeline():
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                tools_response = await session.list_tools()
                all_mcp_schemas = {t["function"]["name"]: t for t in format_mcp_tools(tools_response.tools)}
                question = (
                    "Сформируй сводный аналитический отчёт по всем сохранённым анализам системных промптов. "
                    "Сохрани отчёт под именем 'analysis_report'."
                )
                trace = await run_agent(
                    question=question, session=session, all_mcp_schemas=all_mcp_schemas,
                    required_tool="save_report", max_steps=15, verbose=True,
                )
                print("\n" + trace.summary())
                return trace

trace_report = asyncio.get_event_loop().run_until_complete(run_analysis_report_pipeline())


  🔧 [1] read_skill({"name": "analysis_report"})
  🔓 Разблокированы MCP: ['get_issue', 'search_users', 'add_issue_comment', 'create_pull_request_review', 'list_commits', 'update_pull_request_branch', 'list_pull_requests', 'search_code', 'create_branch', 'get_pull_request_status', 'get_pull_request_files', 'create_pull_request', 'create_or_update_file', 'push_files', 'create_issue', 'get_pull_request', 'merge_pull_request', 'create_repository', 'update_issue', 'search_issues', 'list_issues', 'fork_repository', 'get_pull_request_reviews', 'get_pull_request_comments', 'get_file_contents', 'search_repositories']
  🔧 [2] list_analyses({})
  🔧 [3] read_analysis({"name": "claude-code"})
  🔧 [3] read_analysis({"name": "claude-cowork"})
  🔧 [3] read_analysis({"name": "claude-desktop-code"})
  🔧 [4] save_report({"name": "analysis_report", "content": "# Аналитический отчёт по системным промптам\n\n## Общая стат)

✅ Агент завершил работу за 5 шагов
💬 Отчёт сохранён. Готов к генерации промпта.

Вопр

In [31]:
TARGET_ASSISTANT = "ассистент для code review, который проверяет Pull Request-ы"

async def run_prompt_generator_pipeline(target: str):
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                tools_response = await session.list_tools()
                all_mcp_schemas = {t["function"]["name"]: t for t in format_mcp_tools(tools_response.tools)}
                question = (
                    f"На основе аналитического отчёта сгенерируй новый системный промпт для: {target}. "
                    f"Применяй лучшие практики из отчёта. "
                    f"Сохрани результат через save_report под именем 'generated_prompt'."
                )
                trace = await run_agent(
                    question=question, session=session, all_mcp_schemas=all_mcp_schemas,
                    required_tool="save_report", max_steps=12, verbose=True,
                )
                print("\n" + trace.summary())
                generated = read_report("generated_prompt")
                print("\n" + "=" * 50)
                print("📄 СГЕНЕРИРОВАННЫЙ ПРОМПТ:")
                print("=" * 50)
                print(generated)
                return trace

trace_generator = asyncio.get_event_loop().run_until_complete(run_prompt_generator_pipeline(TARGET_ASSISTANT))


  🔧 [1] read_skill({"name": "prompt_generator"})
  🔓 Разблокированы MCP: ['get_issue', 'search_users', 'add_issue_comment', 'create_pull_request_review', 'list_commits', 'update_pull_request_branch', 'list_pull_requests', 'search_code', 'create_branch', 'get_pull_request_status', 'get_pull_request_files', 'create_pull_request', 'create_or_update_file', 'push_files', 'create_issue', 'get_pull_request', 'merge_pull_request', 'create_repository', 'update_issue', 'search_issues', 'list_issues', 'fork_repository', 'get_pull_request_reviews', 'get_pull_request_comments', 'get_file_contents', 'search_repositories']
  🔧 [2] read_report({"name": "analysis_report"})
  🔧 [3] save_report({"name": "generated_prompt", "content": "# Системный промпт: Code Review Assistant\n\n## Роль\nАссис)

✅ Агент завершил работу за 4 шагов
💬 ✅ Промпт сгенерирован и сохранён.
Оценка по критериям:
- Role/Persona: присутствует
- Few-shot: 2 примера
- Chain-of-Thought: присутствует
- Output Format: присутствует
- Guar

<img src="pictures\hw_file_1.PNG" width="600"/>

---
## Бонус: параллельный анализ через субагентов (критерии 9–10)

Каждый файл анализируется в отдельном агентном вызове с чистым контекстом (`messages` + `AgentTrace`), запущенном параллельно через `asyncio.gather`.


In [32]:
async def analyze_single_file(file_info: dict, session, all_mcp_schemas: dict) -> AgentTrace:
    """Субагент: анализирует один файл с чистым контекстом."""
    question = (
        f"Проанализируй системный промпт из файла '{file_info['path']}' "
        f"в репозитории asgeirtj/system_prompts_leaks. "
        f"Сохрани анализ под именем '{file_info['name'].replace('.md', '')}'"
    )
    return await run_agent(
        question=question, session=session, all_mcp_schemas=all_mcp_schemas,
        required_tool="save_analysis", max_steps=12, verbose=False,
    )


async def run_parallel_analysis_pipeline():
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                tools_response = await session.list_tools()
                all_mcp_schemas = {t["function"]["name"]: t for t in format_mcp_tools(tools_response.tools)}
                print("📂 Получаем список файлов...")
                files = await list_repo_files(session)
                files_to_analyze = files[:MAX_FILES_TO_ANALYZE]
                print(f"🚀 Запускаем {len(files_to_analyze)} субагентов параллельно...\n")
                tasks = [analyze_single_file(f, session, all_mcp_schemas) for f in files_to_analyze]
                traces = await asyncio.gather(*tasks)
                print(f"\n{'=' * 50}")
                print("✅ Параллельный анализ завершён")
                for i, (fi, tr) in enumerate(zip(files_to_analyze, traces), 1):
                    ok = "✅" if tr.total_steps < 12 else "⚠\ufe0f"
                    print(f"  {ok} [{i}] {fi['name']} — {tr.total_steps} шагов, {tr.total_tokens_total} токенов")
                print(f"\n{list_analyses()}")
                return traces

traces_parallel = asyncio.get_event_loop().run_until_complete(run_parallel_analysis_pipeline())


📂 Получаем список файлов...
🚀 Запускаем 3 субагентов параллельно...


✅ Параллельный анализ завершён
  ✅ [1] claude-code.md — 4 шагов, 53474 токенов
  ✅ [2] claude-cowork.md — 4 шагов, 93574 токенов
  ✅ [3] claude-desktop-code.md — 4 шагов, 41002 токенов

📋 Сохранённые анализы (3):
  • claude-code (2226 байт)
  • claude-cowork (2061 байт)
  • claude-desktop-code (1996 байт)


<img src="pictures\hw_file_2.PNG" width="600"/>